# Instancia Premier League 2021-2022

Descarga desde Football-Data la base de esta temporada y la usa como conjunto de equipos para resolver un calendario doble Round-Robin nuevo con el modelo compartido `modelo_calendario.py`.

In [ ]:
# Evaluar la instancia 2021-2022 con GurobiPy en dos configuraciones.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2021-2022"
URL_CSV = "https://www.football-data.co.uk/mmz4281/2122/E0.csv"
PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "tiempo_limite": 3600,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}
resultados = {
    nombre: resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        **PARAMETROS_MODELO,
    )
    for nombre, desactivar_cortes_heuristicas in configuraciones.items()
}

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
display(reporte)

resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos históricos descargados: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))